<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>


# Substantia Nigra Cell Subtyping 
## Case Study — Part 02: Preprocessing and Feature Selection

**ASAP CRN Learning Lab**  
Reproducible exploratory and meta-analysis examples using ASAP CRN data

---

### Overview

This notebook continues the Substantia Nigra case study by applying standardized preprocessing and feature selection to the AnnData artifacts generated in Part 01. These steps prepare the data for downstream integration, clustering, and cell-type annotation.

---

### Learning Objectives

By the end of this notebook, you will be able to:

- Normalize and log-transform single-cell expression data
- Identify and subset highly variable genes (HVGs)
- Generate an analysis-ready AnnData object for downstream workflows
---


### Prerequisites

- Completion of Case Study — Part 01: Set Up and Data Preparation
- Access to the previously generated Substantia Nigra AnnData artifacts
- A configured analysis environment with required dependencies available
---

### Inputs

- Curated Substantia Nigra AnnData object (full gene space)
    - Example: `asap-{dataset_team}__sn_cells__full_genes__curated.h5ad`
---

### Outputs

This notebook generates the following AnnData artifacts:

- **Analysis-ready Substantia Nigra AnnData objectv**
    - Example: asap-{dataset_team}__sn_cells__preprocessed.h5ad
    - Contains:
        - Substantia Nigra–derived cells
        - Normalized and log-transformed expression values
        - Preprocessing outputs (e.g., PCA stored in .obsm)ebooks  

> This artifact is designed to be reused directly in downstream notebooks focused on integration, clustering, and cell-type annotation.
---

### Notes

- Parameters may be adapted for exploratory analyses; however, downstream notebooks assume the default outputs generated here.


## Table of Contents

1. [Package Imports and Configuration](#2-package-imports-and-configuration)
2. [Data Sources and Context](#3-data-sources-and-context)
4. [Data Preproccesing](#4-data-preprocessing)
5. [Data Export](#5-data-export)

## 1. Package Imports and Configuration

In [1]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


time: 139 μs (started: 2026-04-06 20:03:39 +00:00)


## 2. Data sources and Context

### 2.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [2]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

Home directory:      /home/jupyter
Workspace root:      /home/jupyter/workspace
Data directory:      /home/jupyter/workspace/Data
ws_files directory:  /home/jupyter/workspace/ws_files

Contents of workspace root:
 - ws_files /
 - 10_Other /
 - 02_Mouse /
 - release_resources /
 - Documentation /
 - 01_PMDBS /
time: 2.26 ms (started: 2026-04-06 20:03:39 +00:00)


In [3]:
## Build and set path to desired dataset

DATASETS_PATH = WS_ROOT / "01_PMDBS"

workflow       = "pmdbs_sc_rnaseq"
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"asap-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH /  dataset_type / bucket_name / workflow
print("Dataset Path:", dataset_path)

# define the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Define the local path for case study output
local_data_path = WS_FILES / "sn_celltyping"
!ls {local_data_path}

Dataset Path: /home/jupyter/workspace/01_PMDBS/sc-rnaseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq
asap-cohort-dataset-metadata.csv	       asap-cohort.sn_hvg.h5ad
asap-cohort.final.h5ad			       mapmycells
asap-cohort.final_metadata.csv		       output_plots
asap-cohort.merged_cleaned_unfiltered.h5ad     output_tables
asap-cohort.sn_cells__full_genes_curated.h5ad  resources
asap-cohort_sn_cells__preprocessed.h5ad
time: 1.09 s (started: 2026-04-06 20:03:39 +00:00)


## 3. Data Preprocessing

In this section, we load the Substantia Nigra–restricted AnnData object generated in Part 01 and prepare it for feature selection. We explicitly preserve raw counts, apply normalization and log-transformation, and compute highly variable genes (HVGs) using a sample-aware strategy.

### 3.1 Load Substantia Nigra AnnData Object

In [4]:
# Load curated Substantia Nigra AnnData object (full gene space)
sn_full_raw_filename = (
    local_data_path / f"asap-{dataset_team}.sn_cells__full_genes_curated.h5ad"
)
adata = sc.read_h5ad(sn_full_raw_filename)

time: 12min 23s (started: 2026-04-06 20:03:40 +00:00)


### 3.2 Preserve Raw Counts and Expression State
We explicitly store raw counts and the unprocessed expression matrix to support reproducibility and downstream reuse.

In [5]:
# Preserve raw counts and original expression matrix
adata.layers['counts'] = adata.X.copy()
adata.raw = adata.copy()

time: 12.7 s (started: 2026-04-06 20:16:04 +00:00)


### 3.3 Normalize and Log-Transform Expression
Expression values are normalized to a fixed library size and log-transformed. These transformed values are used for HVG detection and downstream dimensionality reduction, while raw counts remain accessible.

In [6]:
# Normalize and log-transform expression values
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
# Save the normalized-log data
adata.layers['lognorm']=adata.X.copy() 

time: 24.9 s (started: 2026-04-06 20:16:17 +00:00)


### 3.4 Recompute Feature Selection and PCA
Because this dataset was generated by subsetting Substantia Nigra cells from a larger whole–brain dataset while retaining the full gene space, the variance structure of the data changes. Highly variable genes and principal components computed on the full dataset reflect global brain-wide variation and are therefore not appropriate for analyses restricted to the Substantia Nigra subset. For this reason, both HVG selection and PCA are recomputed to capture biologically meaningful variation specific to Substantia Nigra cells.

In [7]:
# Detect highly variable genes (HVGs) within the Substantia Nigra subset
n_top_genes = 3000
hvgs_res =sc.experimental.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        batch_key="sample", #using sample not batch
        flavor="pearson_residuals",
        check_values=True,
        layer="counts",
        subset=False,
        inplace=True
)

time: 1min 11s (started: 2026-04-06 20:16:42 +00:00)


In [8]:
# Double check that no transcripts not found in cells are in the atlas
min_cells = 5
sc.pp.filter_genes(adata, min_cells=min_cells)

time: 56.7 s (started: 2026-04-06 20:17:53 +00:00)


In [9]:
# Build a kNN graph in the scVI latent space to capture denoised,
# batch-corrected cellular relationships prior to Leiden clustering.
sc.pp.neighbors(adata, use_rep="_X_scVI", n_neighbors=20, method = "umap", transformer='pynndescent',  key_added='neighbors_scvi')
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_sn_scVI_1.0", neighbors_key='neighbors_scvi',random_state=0)

/opt/conda/envs/sn_celltyping/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_3877/2558793904.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=1.0, key_added="leiden_sn_scVI_1.0", neighbors_key='neighbors_scvi',random_state=0)


time: 29min 11s (started: 2026-04-06 20:18:50 +00:00)


> ⏱️ **Expected runtime:** ~30 minutes depending on dataset size and available compute when run for the first time.

In [10]:
adata

AnnData object with n_obs × n_vars = 692895 × 34434
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'total_counts', 'total_counts_rb', 'pct_counts_rb', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'sample', 'batch', 'team', 'dataset', 'batch_id', 'S_score', 'G2M_score', 'phase', 'brain_region', 'brain_region_simple', 'case_id', 'condition_id', 'region_level_1', 'region_level_2', 'dataset_id', 'biobank_name', '_cell_type', '_phenotype', '_rho', '_prob', '_class_name', '_subclass_name', '_supertype_name', '__scvi_batch', '__scvi_labels', '_C_scANVI', '_leiden_res_0.05', '_leiden_res_0.10', '_leiden_res_0.20', '_leiden_res_0.40', 'leiden_sn_scVI_1.0'
    var: 'feature_type', 'genome', 'gene_id', 'mt', 'rb', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_nbatches', 'highly_variable_intersection', 'highly_variable', 'n_cells'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'estimator', 'fr

time: 5.29 ms (started: 2026-04-06 20:48:02 +00:00)


## 4. Export Data
In this step, we export the analysis-ready Substantia Nigra AnnData object generated in this notebook. This artifact reflects all preprocessing, feature selection, and dimensionality reduction steps applied after subsetting to Substantia Nigra cells.

The exported AnnData object serves as a reproducible input for subsequent parts of the case study and supports downstream integration, clustering, and cell-type annotation workflows.

This AnnData object contains:

- Cells restricted to the Substantia Nigra (SN)
- Expression values that have been normalized and log-transformed
- Highly variable genes (HVGs) used for downstream modeling
- Raw counts preserved in .layers["counts"]
- Preprocessing outputs (e.g., PCA, latent embeddings) stored in .obsm
- Curated metadata and annotations stored in .obs

In [19]:
# Drop large neighbor matrices
for k in ['neighbors_scvi_distances','neighbors_scvi_connectivities']:
    adata.obsp.pop(k, None)

# Drop heavy per-gene stats (keep only highly_variable + n_cells)
adata.var.drop(columns=[
    'means','variances','residual_variances',
    'highly_variable_rank','highly_variable_nbatches','highly_variable_intersection'
], errors='ignore', inplace=True)


time: 3.16 ms (started: 2026-04-06 21:08:58 +00:00)


In [18]:
sn_processed_filename = (
    local_data_path / f"asap-{dataset_team}_sn_cells__preprocessed.h5ad"
)

adata.write_h5ad(sn_processed_filename)

time: 15min 28s (started: 2026-04-06 20:53:29 +00:00)


> ⏱️ **Expected runtime:** ~20 minutes depending on dataset size and available compute when run for the first time.

---

## Summary and Next Steps

In this notebook, we prepared Substantia Nigra–restricted single-cell data for downstream analysis by applying standardized preprocessing, feature selection, and dimensionality reduction. These steps ensure that subsequent analyses capture biologically meaningful variation specific to Substantia Nigra cells.

The exported AnnData artifact provides a stable, reproducible handoff for downstream workflows, including integration, clustering, and cell-type annotation.

### Continue the Case Study

- Proceed to **Part 03: MapMyCells Mapping**
- Refer to the **ASAP-CRN Learning Lab documentation** for additional context, workflows, and best practices.

> This notebook is part of the ASAP-CRN Learning Lab and is intended to be executed using approved data accessed through the ASAP-CRN Cloud.

